In [ ]:

# Reproducibility
import numpy as np
import random
rng = np.random.default_rng(seed=511)
random.seed(511)



# Timing and plotting
import time
import matplotlib.pyplot as plt

# Utility: timing helper
def time_function(fn, *args, repeat=5, **kwargs):
    """Return the best of `repeat` wall-clock timings for fn(*args, **kwargs)."""
    best = float('inf')
    result = None
    for _ in range(repeat):
        start = time.perf_counter()
        out = fn(*args, **kwargs)
        elapsed = time.perf_counter() - start
        if elapsed < best:
            best = elapsed
            result = out
    return best, result

print("Environment ready.")





  def insertion_sort(arr):
    a = list(arr)

    for i in range(1, len(a)):
        key = a[i]
        j = i - 1

        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j = j - 1

        a[j + 1] = key

    return a



# Experiment sizes (adjust if runtime too long or too short); you can also add more data points
sizes = [5_000, 10_000, 20_000]
repeat = 3

# Cap insertion sort at manageable n
ins_cap = 20_000




results = {
    'n': [],
    'sorted_builtin': [],
    'insertion_sort': [],
    'numpy_sort': []
}

for n in sizes:

    data = rng.integers(
        low=0,
        high=10_000_000,
        size=n,
        dtype=np.int64
    )

    # Python sorting
    t_sorted, _ = time_function(
        sorted, data, repeat=repeat
    )

    # NumPy sorting
    t_numpy, _ = time_function(
        np.sort, data, repeat=repeat
    )

    # Insertion sorting
    if n <= ins_cap:
        t_insert, _ = time_function(
            insertion_sort, data, repeat=repeat
        )
    else:
        t_insert = np.nan

    # Save results
    results['n'].append(n)
    results['sorted_builtin'].append(t_sorted)
    results['insertion_sort'].append(t_insert)
    results['numpy_sort'].append(t_numpy)

import pandas as pd

dfA = pd.DataFrame(results)

dfA



#TODO; plot linear plot from results using the log log template to help.  Need two plots for full credit

plt.figure()
for key in ['sorted_builtin', 'numpy_sort', 'insertion_sort']:
    plt.plot(dfA['n'], dfA[key], marker='o', label=key)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('n (log)')
plt.ylabel('Runtime (s, log)')
plt.title('Sorting runtime vs n (log-log)')
plt.legend()
plt.show()



# Linear plot

plt.figure()

for key in ['sorted_builtin', 'numpy_sort', 'insertion_sort']:
    plt.plot(dfA['n'], dfA[key], marker='o', label=key)

plt.xlabel('Input size (n)')
plt.ylabel('Runtime (seconds)')

plt.title('Sorting Runtime vs Input Size')

plt.legend()
plt.grid(True)

plt.show()



sizes_B = [10_000, 100_000, 500_000]
queries_per_size = 1_000

def gen_ids(n):
    ids = [f"id_{i:07d}" for i in range(n)]
    rng.shuffle(ids)
    return ids

def membership_timing(ids, structure_type='list'):
    if structure_type == 'list':
        coll = list(ids)
        def contains(x): return x in coll
    elif structure_type == 'set':
        coll = set(ids)
        def contains(x): return x in coll
    elif structure_type == 'dict':
        coll = {x: True for x in ids}
        def contains(x): return x in coll

    present = rng.choice(ids, size=queries_per_size//2, replace=False)
    absent = [f"nope_{i:07d}" for i in range(queries_per_size - len(present))]
    queries = np.concatenate([present, absent])
    rng.shuffle(queries)

    start = time.perf_counter()
    for q in queries:
        _ = contains(q)
    elapsed = time.perf_counter() - start
    return elapsed

records = {'n': [], 'structure': [], 'time_s': []}

for n in sizes_B:
    ids = gen_ids(n)
    for stype in ['list', 'set', 'dict']:
        t = membership_timing(ids, structure_type=stype)
        records['n'].append(n)
        records['structure'].append(stype)
        records['time_s'].append(t)

dfB = pd.DataFrame(records)
dfB




plt.figure()
for stype in ['list', 'set', 'dict']:
    sub = dfB[dfB['structure'] == stype]
    plt.plot(sub['n'], sub['time_s'], marker='o', label=stype)
plt.xlabel('n')
plt.ylabel('Total time for membership queries (s)')
plt.title('Membership testing time vs n')
plt.legend()
plt.show()
